In [ ]:
import sys
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis")
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis\ucla-lapd")
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, interpolate, ndimage
from scipy.fft import next_fast_len

import read_hdf5_bapsflib as rh
from bapsflib import lapd

from lp_analysis import find_sweep_indices, reshape_IV
from lp_iv_analysis import analyze_IV
from Mar2026_mach import  load_mach_envelope_data, plot_mach_heatmap


%matplotlib qt
plt.rcParams.update({'font.size': 16})

In [ ]:
ifn = r"D:\data\LAPD\Mar26-data\03-mach-emissive-xline-bias.hdf5"

# Extract directory and run number from ifn
data_dir = os.path.dirname(ifn)
run_num = os.path.basename(ifn).split('-')[0]  # "11" from "11-lp-sweep-xyplane-bias.hdf5"

with lapd.File(ifn) as f:
    rh.show_info(f)
    adc, digi_dict = rh.read_digitizer_config(f)
    pos_dict, xpos, ypos, zpos, npos, nshot = rh.read_probe_motion_bmotion(f)
    key = list(pos_dict.keys())[0]
    pos_array = pos_dict[key]
    nx = len(np.unique(xpos))
    ny = len(np.unique(ypos))

In [61]:
with lapd.File(ifn) as f:
    data, tarr = rh.read_data(f, 1, 1, index_arr=slice(npos*nshot, None), adc=adc)
    Vx_arr = data['signal'].reshape((npos, nshot, -1))
    data, tarr = rh.read_data(f, 1, 2, index_arr=slice(npos*nshot, None), adc=adc)
    Vy_arr = data['signal'].reshape((npos, nshot, -1))
nt = len(tarr)

Vx_arr_filt = ndimage.gaussian_filter1d(Vx_arr, 50, axis=-1)
Vy_arr_filt = ndimage.gaussian_filter1d(Vy_arr, 50, axis=-1)

c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\maps\digitizers\siscrate.py:728: UserWarning: HDF5 structure unexpected...'siscf-5ch-mach-emi/SIS crate 3302 configurations[1]' does not define any valid channel numbers...not adding to `configs` dict
  warn(why)
c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\maps\digitizers\siscrate.py:728: UserWarning: HDF5 structure unexpected...'siscf-5ch-mach-emi/SIS crate 3302 configurations[2]' does not define any valid channel numbers...not adding to `configs` dict
  warn(why)
c:\Users\hjia9\Documents\GitHub\data-analysis\.venv\Lib\site-packages\bapsflib-0.0.0-py3.11.egg\bapsflib\_hdf\utils\hdfreaddata.py:508: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  data = np.empty(shape, dtype=dtype)
c:\Users\hjia9\Doc

In [49]:
Vx_arr_filt.shape

(81, 5, 108544)

In [59]:
plt.figure()
for i in [0,40,80]:
    sig = Vx_arr[i, 0]
    filt = ndimage.gaussian_filter1d(sig, 50)
    plt.plot(tarr, sig, alpha=0.5)
    plt.plot(tarr, filt, label=f"Position {pos_array[i]}")

plt.xlabel("Time")
plt.ylabel("Voltage")
plt.legend()
plt.show()

In [62]:
Vx_arr_avg = Vx_arr_filt.mean(axis=1)

In [63]:
# Separate Vx_arr_avg into 20 time segments and plot voltage vs xpos
num_segments = 20
segment_len = nt // num_segments

# Create 4 panels with 5 segments each
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
axs = axs.flatten()

colors = plt.cm.viridis(np.linspace(0, 1, 5))

for panel in range(4):
    ax = axs[panel]
    
    # Plot 5 segments per panel
    for seg_in_panel in range(5):
        seg_idx = panel * 5 + seg_in_panel
        
        # Get the time index for this segment (use middle of segment)
        time_idx = (seg_idx + 0.5) * segment_len
        time_idx = int(time_idx)
        time_idx = min(time_idx, nt - 1)  # Ensure we don't exceed array bounds
        
        # Get voltage at this time, reshaped to spatial positions
        voltage = Vx_arr_avg[:, time_idx]
        
        # Time value in milliseconds
        t_val = tarr[time_idx] * 1e3
        
        # Plot voltage vs position
        ax.plot(xpos, voltage, 'o-', color=colors[seg_in_panel], 
                label=f't = {t_val:.2f} ms', linewidth=2, markersize=4)
    
    ax.set_xlabel('X Position [cm]', fontsize=11)
    ax.set_ylabel('Vx [V]', fontsize=11)
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Test on one signal
sig = ndimage.gaussian_filter1d(sig, 50)
# ==========================================
# 1. Parameters
# ==========================================
fs = 1.0e6               
num_timepoints = 108544
median_window = 5001     # Large window 
dec_factor = 100          # Downsample factor
# ==========================================
# 2. Method A: Full Resolution (Brute Force)
# ==========================================
print("Running Full Resolution Median Filter...")
start_time_full = time.time()

# 1D median filter
baseline_full = ndimage.median_filter(sig, size=median_window)

end_time_full = time.time()
time_full = end_time_full - start_time_full
print(f"Full Resolution took: {time_full:.4f} seconds")

# ==========================================
# 3. Method B: Decimate & Interpolate
# ==========================================
print("Running Decimate & Interpolate Filter...")
start_time_fast = time.time()

# 1. Decimate
x_orig = np.arange(num_timepoints)
x_dec = np.arange(0, num_timepoints, dec_factor)
signal_dec = sig[::dec_factor]

# 2. Scale window
small_window = median_window // dec_factor
if small_window % 2 == 0: small_window += 1

# 3. Median filter on small array
baseline_dec = ndimage.median_filter(signal_dec, size=small_window)

# 4. Interpolate
baseline_fast = interpolate.interp1d(x_dec, baseline_dec, fill_value="extrapolate")(x_orig)

end_time_fast = time.time()
time_fast = end_time_fast - start_time_fast
print(f"Decimate & Interpolate took: {time_fast:.4f} seconds")

# ==========================================
# 4. Calculate Difference
# ==========================================
speedup = time_full / time_fast if time_fast > 0 else float('inf')
print(f"--> Speedup Factor: {speedup:.1f}x faster")

difference = np.abs(baseline_full - baseline_fast)
max_error = np.max(difference)
print(f"--> Maximum amplitude difference: {max_error:.6f}")

# ==========================================
# 5. Visualization
# ==========================================
plt.figure(figsize=(14, 8))

# Top Plot: Overlay of both baselines
plt.subplot(2, 1, 1)
plt.plot(tarr, sig, color='gray', alpha=0.4, label='Raw Signal')
plt.plot(tarr, baseline_full, color='blue', linewidth=3, label='Full Resolution Baseline')
plt.plot(tarr, baseline_fast, color='red', linestyle='--', linewidth=2, label='Interpolated Baseline')
plt.title(f'Baseline Comparison (Speedup: {speedup:.1f}x)')
plt.ylabel('Amplitude')
plt.legend()
plt.grid(True)

# Bottom Plot: The Error (Difference)
plt.subplot(2, 1, 2)
plt.plot(tarr, difference, color='purple', label='Absolute Difference |Full - Interpolated|')
plt.title(f'Interpolation Error (Max Error: {max_error:.6f})')
plt.xlabel('Time (Seconds)')
plt.ylabel('Error Amplitude')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
fs = 1.0e6
median_window = 5001     
dec_factor = 100
lowcut = 10.0           
highcut = 10000.0        
filter_order = 4

num_samples = len(sig)
t = np.arange(num_samples) / fs

# ==========================================
# 2. Optimized Baseline Subtraction
# ==========================================
sig_dec = sig[::dec_factor]
x_orig = np.arange(num_samples)
x_dec = np.arange(0, num_samples, dec_factor)

small_win = (median_window // dec_factor) // 2 * 2 + 1 
baseline_dec = ndimage.median_filter(sig_dec, size=small_win)

itp = interpolate.interp1d(x_dec, baseline_dec, kind='linear', fill_value="extrapolate")
baseline = itp(x_orig)

sig_flattened = sig - baseline

# ==========================================
# 3. Filtering (Using Stable SOS) & Envelope
# ==========================================
nyq = 0.5 * fs

# CRITICAL FIX: Use output='sos' instead of 'ba' to prevent numerical explosion
sos = signal.butter(filter_order, [lowcut/nyq, highcut/nyq], btype='band', output='sos')

# Use sosfiltfilt instead of filtfilt
sig_filtered = signal.sosfiltfilt(sos, sig_flattened)

rms_window_samples = 500 


sig_squared = sig_filtered ** 2
mean_squares = ndimage.uniform_filter1d(sig_squared, size=rms_window_samples)
rms_envelope = np.sqrt(mean_squares)
envelope = rms_envelope * np.sqrt(2)

# ==========================================
# 4. Final Debug Plot
# ==========================================
fig, ax = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Plot 1: Raw vs Baseline
ax[0].plot(t, sig, color='gray', alpha=0.5, label='Raw 1D Signal')
ax[0].plot(t, baseline, color='red', label='Interpolated Baseline')
ax[0].set_title("Step 1: Baseline Extraction")
ax[0].legend()

# Plot 2: Flattened vs Filtered
ax[1].plot(t, sig_flattened, color='lightgray', label='Flattened (Raw - Base)')
ax[1].plot(t, sig_filtered, color='blue', alpha=0.8, label='Stable Bandpass Filtered')
ax[1].set_title("Step 2: Filtering (Numerical Stability Fixed)")
ax[1].set_ylabel("Amplitude")
ax[1].legend()

# Plot 3: Resulting Envelope
ax[2].plot(t, sig_filtered, color='blue', alpha=0.3, label='Filtered Signal')
ax[2].plot(t, envelope, color='darkorange', linewidth=2, label='Envelope')
ax[2].set_title("Step 3: Final Envelope Detection")
ax[2].set_xlabel("Time (s)")
ax[2].set_ylabel("Amplitude")
ax[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
envelope_save_path = os.path.join(data_dir, f"{run_num}-mach-envelope-data.npz")
result = load_mach_envelope_data(envelope_save_path)

tarr = result['tarr_downsampled']
nt = len(tarr)
nx = len(xpos)
ny = len(ypos)

vx_arr = result['heatmap_Vx']
vy_arr = result['heatmap_Vy']

heat_map_Vx = vx_arr.reshape((ny, nx, nt))
heat_map_Vy = vy_arr.reshape((ny, nx, nt))

In [ ]:
num_parts = 10
part_len = nt // num_parts
avg_vx = np.zeros((npos, num_parts))
for part in range(num_parts):
    start = part * part_len
    end = (part + 1) * part_len if part < num_parts - 1 else nt
    avg_vx[:, part] = np.mean(vx_arr[:, start:end], axis=1)

fig, ax = plt.subplots(2, 1, figsize=(10, 8))
ax[0].set_title("Mach Probe Voltage vs. Time")
for i in [0, 500, 1000, 1500]:
    ax[0].plot(tarr, vx_arr[i], label=f"Position {pos_array[i]}")
ax[0].set_xlabel("Time [s]")
ax[0].set_ylabel("Vx [V]")
ax[0].legend()

ax[1].set_title("Averaged Vx over 10 Time Parts")
for i in [0, 500, 1000, 1500]:
    ax[1].scatter(range(num_parts), avg_vx[i], label=f"Position {pos_array[i]}")
ax[1].set_xlabel("Time Part")
ax[1].set_ylabel("Averaged Vx [V]")
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate time bin size
num_parts = 10
part_len = nt // num_parts

# Define spatial extent
extent = (xpos.min(), xpos.max(), ypos.min(), ypos.max())

# one shared color range for both fields
vmin = 0
vmax = 0.02

# 2 rows (Vx, Vy), num_parts columns (long skinny)
fig, axs = plt.subplots(2, num_parts, figsize=(1.5*num_parts, 6), squeeze=False)

for part in range(num_parts):
    start_idx = part * part_len
    end_idx = (part + 1) * part_len if part < num_parts - 1 else nt
    t_center = tarr[start_idx:end_idx].mean()
    vx_avg = heat_map_Vx[:, :, start_idx:end_idx].mean(axis=2)
    vy_avg = heat_map_Vy[:, :, start_idx:end_idx].mean(axis=2)

    ax_vx = axs[0, part]
    ax_vy = axs[1, part]

    im = ax_vx.imshow(vx_avg, origin='lower', cmap='magma',
                      extent=extent, interpolation='gaussian',
                      vmin=vmin, vmax=vmax)
    ax_vx.set_title(f"{t_center*1e3:.2f} ms", fontsize=8)
    ax_vx.set_xticks([])
    ax_vx.set_yticks([])

    ax_vy.imshow(vy_avg, origin='lower', cmap='magma',
                 extent=extent, interpolation='gaussian',
                 vmin=vmin, vmax=vmax)


# shared row labels
fig.text(0.01, 0.72, "Vx", va="center", ha="left", fontsize=12, weight="bold")
fig.text(0.01, 0.28, "Vy", va="center", ha="left", fontsize=12, weight="bold")

fig.subplots_adjust(left=0.06, right=0.98, top=0.92, bottom=0.15,
                    wspace=0.12, hspace=0.12)

# horizontal shared colorbar at bottom
cbar_ax = fig.add_axes([0.15, 0.06, 0.70, 0.02])
fig.colorbar(im, cax=cbar_ax, orientation='horizontal',
             label='Voltage amplitude (V)')

plt.show()
